# Vector Ideal Versus Current-Lab Case 1

The current laboratory vector case is not a claim of full true radial or azimuthal vector-beam generation. It is a limited current-lab SOP-encoded approximation unless additional polarisation-conversion or independent-axis hardware is introduced.


In [1]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_vector, vbb_style
from vbb_study.publication import vector as vector_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_csv = PATHS["csv"] / "vector"
out_fig = PATHS["figures"] / "vector"
compat_csv = PATHS["csv"] / "publication_study"
out_csv.mkdir(parents=True, exist_ok=True)
out_fig.mkdir(parents=True, exist_ok=True)
compat_csv.mkdir(parents=True, exist_ok=True)
vbb_style.apply_style()

cfg = bt.default_config(PRESET)
design = bt.compute_design_from_targets(cfg.laser, cfg.target, cfg.material)
grid = bt.make_xy_grid(256, 0.18 * bt.um)
KR = 0.95 / bt.um
WAIST = 48.0 * bt.um
ELL_VALUES = (1, 3)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides to `cfg` before running any study cell below.
from vbb_study.publication import notebook_controls as nb_controls
from vbb_study.config import um as _um

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='vector',
    # ── edit these to override the base configuration ───────────────────────
    ell=1,
    target_core_diameter_um=3.0,
    target_bessel_length_um=150.0,
    objective_NA=0.45,
)

# Wire control parameters into `cfg` so downstream cells use them.
_p = NOTEBOOK_CONTROLS.parameters or {}
if "ell" in _p:
    cfg = replace(cfg, target=replace(cfg.target, ell=int(_p["ell"])))
if "target_core_diameter_um" in _p:
    cfg = replace(cfg, target=replace(cfg.target, target_core_diameter_m=float(_p["target_core_diameter_um"]) * _um))
if "target_bessel_length_um" in _p:
    cfg = replace(cfg, target=replace(cfg.target, target_bessel_length_m=float(_p["target_bessel_length_um"]) * _um))
if "objective_NA" in _p:
    cfg = replace(cfg, objective=replace(cfg.objective, NA=float(_p["objective_NA"])))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved.
from vbb_study.publication import notebook_widgets as nbw

_panel = nbw.interactive_quicklook(cfg, method='holographic', preset='fast', ell_range=(0, 6))
display(_panel)


In [2]:
def _stamp(row):
    vector_schema.annotate_vector_row(row, run_id=RUN_ID, qa_status="exploratory")
    return row

def _scalar_metrics():
    small_grid = replace(cfg.grid, ideal_N=160, N=160, crop_pixels=96, axial_points=7, coarse_scan_points=7)
    run_cfg = replace(cfg, grid=small_grid, target=replace(cfg.target, ell=1))
    z_values = np.linspace(0.0, 100.0 * bt.um, 7)
    result = bt.run_case(run_cfg, preset=PRESET, path="ideal", case_id="vector_scalar_reference_ell1", z_values_m=z_values)
    return result["metrics"]

scalar_metrics = _scalar_metrics()

def _row_from_case(case_id, case, *, vector_mode, vector_model, model_level, generation_method, lab_realizable, simulation_only, requires_element, vector_method, vector_encoder_hardware, uses_waveplates, uses_two_slm, uses_shared_director_axis, scalar_reference_case_id="vector_scalar_reference_ell1"):
    row = {
        "case_id": case_id,
        "preset": PRESET,
        "path": "vector_ideal_vs_lab_case1",
        "beam_family": "vector",
        "model_level": model_level,
        "generation_method": generation_method,
        "vector_mode": vector_mode,
        "vector_model": vector_model,
        "vector_program": case.get("target", case.get("mode", vector_mode)),
        "vector_method": vector_method,
        "vector_encoder_hardware": vector_encoder_hardware,
        "lab_realizable": lab_realizable,
        "simulation_only": simulation_only,
        "requires_element": requires_element,
        "uses_waveplates": uses_waveplates,
        "uses_two_slm": uses_two_slm,
        "uses_shared_director_axis": uses_shared_director_axis,
        "encoded_power_fraction": case.get("encoded_power_fraction", np.nan),
        "scalar_reference_case_id": scalar_reference_case_id,
        "ell": int(case.get("ell", 1)),
        "target_core_diameter_um": float(cfg.target.target_core_diameter_m / bt.um),
        "vortex_main_ring_diameter_um": float(2.0 * vbb_vector.predicted_ring_radius(int(case.get("ell", 1)), KR) / bt.um),
        "canonical_zone_um": scalar_metrics.get("canonical_zone_um", scalar_metrics.get("bessel_zone_um")),
        "strict_bessel_region_um": scalar_metrics.get("strict_bessel_region_um", scalar_metrics.get("bessel_region_um")),
        "propagation_power_drift_fraction": scalar_metrics.get("propagation_power_drift_fraction", np.nan),
        "propagation_power_label": scalar_metrics.get("propagation_power_label", "unknown"),
        "hardware_note": case.get("hardware_note", ""),
    }
    if case.get("field") is not None:
        row["total_power_au"] = float(np.sum(case["total_intensity"]) * float(case["grid"]["dx"]) ** 2)
    return _stamp(row)


In [3]:
ideal_radial = vbb_vector.build_analytic_vector_mode(grid, ell=1, kr_m_inv=KR, waist_m=WAIST, mode="radial")
ideal_azimuthal = vbb_vector.build_analytic_vector_mode(grid, ell=1, kr_m_inv=KR, waist_m=WAIST, mode="azimuthal")
case1 = vbb_vector.build_actual_lab_vector_case(
    grid,
    ell=1,
    kr_m_inv=KR,
    waist_m=WAIST,
    target="achievable_sop",
    method="B",
    carrier_lpmm=2.5,
)
paper_replica = vbb_vector.build_analytic_vector_mode(grid, ell=1, kr_m_inv=KR, waist_m=WAIST, mode="radial")

rows = [
    _stamp({
        "case_id": "vector_scalar_reference_ell1",
        "preset": PRESET,
        "path": "vector_ideal_vs_lab_case1",
        "beam_family": "scalar_reference",
        "model_level": "scalar_reference",
        "generation_method": "scalar_reference",
        "vector_mode": "scalar_reference",
        "vector_model": "scalar_sas_with_jones_overlay",
        "vector_program": "scalar_reference",
        "vector_method": "not_applicable",
        "vector_encoder_hardware": "scalar_holographic_reference",
        "lab_realizable": True,
        "simulation_only": False,
        "requires_element": "none",
        "uses_waveplates": False,
        "uses_two_slm": False,
        "uses_shared_director_axis": False,
        "ell": 1,
        "target_core_diameter_um": float(cfg.target.target_core_diameter_m / bt.um),
        "vortex_main_ring_diameter_um": float(2.0 * vbb_vector.predicted_ring_radius(1, KR) / bt.um),
        "canonical_zone_um": scalar_metrics.get("canonical_zone_um", scalar_metrics.get("bessel_zone_um")),
        "strict_bessel_region_um": scalar_metrics.get("strict_bessel_region_um", scalar_metrics.get("bessel_region_um")),
        "propagation_power_drift_fraction": scalar_metrics.get("propagation_power_drift_fraction", np.nan),
        "propagation_power_label": scalar_metrics.get("propagation_power_label", "unknown"),
    }),
    _row_from_case(
        "ideal_radial_target_ell1",
        ideal_radial,
        vector_mode="radial",
        vector_model="ideal_jones_target",
        model_level="ideal_target",
        generation_method="qplate_or_vector_converter",
        lab_realizable=False,
        simulation_only=False,
        requires_element="qplate_or_vector_mode_converter",
        vector_method="analytic_reference",
        vector_encoder_hardware="not_current_bench",
        uses_waveplates=False,
        uses_two_slm=False,
        uses_shared_director_axis=False,
    ),
    _row_from_case(
        "ideal_azimuthal_target_ell1",
        ideal_azimuthal,
        vector_mode="azimuthal",
        vector_model="ideal_jones_target",
        model_level="ideal_target",
        generation_method="qplate_or_vector_converter",
        lab_realizable=False,
        simulation_only=False,
        requires_element="qplate_or_vector_mode_converter",
        vector_method="analytic_reference",
        vector_encoder_hardware="not_current_bench",
        uses_waveplates=False,
        uses_two_slm=False,
        uses_shared_director_axis=False,
    ),
    _row_from_case(
        "current_lab_case1_sop_method_b",
        case1,
        vector_mode="sop_encoded_case1",
        vector_model="current_lab_case1_sop_encoded",
        model_level="current_lab_approximation",
        generation_method="two_slm_same_axis_sop",
        lab_realizable=True,
        simulation_only=False,
        requires_element="none",
        vector_method="method_B_complex_amplitude_proxy",
        vector_encoder_hardware="case1_same_axis_no_waveplates",
        uses_waveplates=False,
        uses_two_slm=True,
        uses_shared_director_axis=True,
    ),
    _row_from_case(
        "paper_replica_radial_diagnostic_ell1",
        paper_replica,
        vector_mode="paper_replica",
        vector_model="paper_replica_baliyan_nishchal",
        model_level="paper_replica",
        generation_method="paper_replica_simulation",
        lab_realizable=False,
        simulation_only=True,
        requires_element="waveplate_chain",
        vector_method="paper_jones_chain",
        vector_encoder_hardware="paper_qwp_hwp_chain",
        uses_waveplates=True,
        uses_two_slm=True,
        uses_shared_director_axis=False,
    ),
]

summary = vector_schema.ordered_vector_frame(rows)
summary.to_csv(out_csv / "vector_ideal_vs_lab_case1_summary.csv", index=False)

compat_ladder = summary.copy()
compat_ladder.to_csv(compat_csv / "stage6_fidelity_ladder_summary.csv", index=False)
compat_ladder.to_csv(out_csv / "stage6_fidelity_ladder_summary.csv", index=False)
summary[summary["case_id"] == "current_lab_case1_sop_method_b"].to_csv(
    compat_csv / "stage6_slm_encoded_vector_summary.csv", index=False
)
summary[summary["case_id"] == "current_lab_case1_sop_method_b"].to_csv(
    out_csv / "stage6_slm_encoded_vector_summary.csv", index=False
)
summary[summary["case_id"] == "paper_replica_radial_diagnostic_ell1"].to_csv(
    compat_csv / "stage6_paper_replica_vector_summary.csv", index=False
)
summary[summary["case_id"] == "paper_replica_radial_diagnostic_ell1"].to_csv(
    out_csv / "stage6_paper_replica_vector_summary.csv", index=False
)

baseline = summary.loc[summary["case_id"] == "vector_scalar_reference_ell1"].iloc[0]
delta_rows = []
for _, row in summary.iterrows():
    if row["case_id"] == "vector_scalar_reference_ell1":
        continue
    delta_rows.append(_stamp({
        "case_id": row["case_id"],
        "preset": PRESET,
        "path": "vector_ideal_vs_lab_case1",
        "beam_family": row["beam_family"],
        "model_level": "diagnostic",
        "generation_method": row["generation_method"],
        "vector_mode": row["vector_mode"],
        "vector_model": "diagnostic_only",
        "vector_program": row["vector_program"],
        "vector_method": row["vector_method"],
        "vector_encoder_hardware": row["vector_encoder_hardware"],
        "lab_realizable": bool(row["lab_realizable"]),
        "simulation_only": bool(row["simulation_only"]),
        "requires_element": row["requires_element"],
        "uses_waveplates": bool(row["uses_waveplates"]),
        "uses_two_slm": bool(row["uses_two_slm"]),
        "uses_shared_director_axis": bool(row["uses_shared_director_axis"]),
        "encoded_power_fraction": row.get("encoded_power_fraction", np.nan),
        "scalar_reference_case_id": "vector_scalar_reference_ell1",
        "ell": int(row["ell"]),
        "canonical_zone_um": row["canonical_zone_um"],
        "strict_bessel_region_um": row["strict_bessel_region_um"],
        "canonical_zone_delta_um": float(row["canonical_zone_um"] - baseline["canonical_zone_um"]),
        "strict_region_delta_um": float(row["strict_bessel_region_um"] - baseline["strict_bessel_region_um"]),
    }))
delta_table = vector_schema.ordered_vector_frame(delta_rows)
delta_table.to_csv(out_csv / "stage6_fidelity_delta_table.csv", index=False)
delta_table.to_csv(compat_csv / "stage6_fidelity_delta_table.csv", index=False)
summary


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,beam_family,model_level,generation_method,hardware_status,...,scalar_reference_case_id,ell,target_core_diameter_um,vortex_main_ring_diameter_um,canonical_zone_um,strict_bessel_region_um,propagation_power_drift_fraction,propagation_power_label,hardware_note,total_power_au
0,20260604T150311Z,2026-06-04T15:04:15.267592+00:00,1.0.0,vector_scalar_reference_ell1,fast,vector_ideal_vs_lab_case1,scalar_reference,scalar_reference,scalar_reference,current_lab_realizable,...,NaN,1,3.0,3.876176,51.83811,33.333333,1.121033,fail,NaN,NaN
1,20260604T150311Z,2026-06-04T15:04:15.267731+00:00,1.0.0,ideal_radial_target_ell1,fast,vector_ideal_vs_lab_case1,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,vector_scalar_reference_ell1,1,3.0,3.876176,51.83811,33.333333,1.121033,fail,,2.269968e-11
2,20260604T150311Z,2026-06-04T15:04:15.267836+00:00,1.0.0,ideal_azimuthal_target_ell1,fast,vector_ideal_vs_lab_case1,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,vector_scalar_reference_ell1,1,3.0,3.876176,51.83811,33.333333,1.121033,fail,,2.269968e-11
3,20260604T150311Z,2026-06-04T15:04:15.267937+00:00,1.0.0,current_lab_case1_sop_method_b,fast,vector_ideal_vs_lab_case1,vector,current_lab_approximation,two_slm_same_axis_sop,current_lab_realizable,...,vector_scalar_reference_ell1,1,3.0,3.876176,51.83811,33.333333,1.121033,fail,Case-1 achievable class: H is shaped and V is ...,4.246733e-09
4,20260604T150311Z,2026-06-04T15:04:15.268020+00:00,1.0.0,paper_replica_radial_diagnostic_ell1,fast,vector_ideal_vs_lab_case1,vector,paper_replica,paper_replica_simulation,simulation_only,...,vector_scalar_reference_ell1,1,3.0,3.876176,51.83811,33.333333,1.121033,fail,,2.269968e-11


In [4]:
fig = vbb_vector.plot_total_and_analyzer_panel(case1["field"], grid, title="Current lab Case 1 SOP-encoded approximation")
vbb_style.save_figure(
    fig,
    out_fig / "vector_current_lab_case1_sop_panel.png",
    "Current laboratory Case 1: limited SOP-encoded approximation using two phase-only SLMs with shared director axis and no waveplates. This is not a claim of true radial or azimuthal vector-beam generation.",
    metadata={"stage": "vector", "figure": "vector_current_lab_case1_sop_panel"},
)
plt.close(fig)

fig = vbb_vector.plot_total_and_analyzer_panel(paper_replica["field"], grid, title="Paper-replica radial diagnostic")
vbb_style.save_figure(
    fig,
    out_fig / "vector_paper_replica_radial_panel.png",
    "Paper-replica Baliyan-Nishchal style radial diagnostic. This route is labelled simulation_only under the current bench assumptions because it requires a waveplate chain not present in Case 1.",
    metadata={"stage": "vector", "figure": "vector_paper_replica_radial_panel"},
)
plt.close(fig)
